<a href="https://colab.research.google.com/github/carlosgmontoya/pytorch_JE/blob/main/John_Elder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# =========================
# 1. Cargar datos (Sin mezclar)
# =========================
ruta_drive = "/content/drive/MyDrive/Colab Notebooks/"

df_train_full = pd.read_csv(ruta_drive + "train.csv")
df_test_final = pd.read_csv(ruta_drive + "test.csv") # El examen final

# 1.1 Limpieza
for df in [df_train_full, df_test_final]:
    if 'datasetId' in df.columns:
        df.drop(columns=['datasetId'], inplace=True)

# 1.2 CAMBIO A BINARIO
# Mantenemos la separación estricta: Train es para aprender, Test es para evaluar
df_train_full['condition'] = df_train_full['condition'].apply(lambda x: 0 if x == 'no stress' else 1)
df_test_final['condition'] = df_test_final['condition'].apply(lambda x: 0 if x == 'no stress' else 1)

# Extraemos valores
X_train_full = df_train_full.drop(columns=["condition"]).values
y_train_full = df_train_full["condition"].values

X_test = df_test_final.drop(columns=["condition"]).values
y_test = df_test_final["condition"].values

print("Modo Riguroso: El set de Test es totalmente independiente.")

# =========================
# 2. División de Validación (Solo del set de entrenamiento)
# =========================
# Usamos el 20% del set de entrenamiento para ir viendo cómo va el modelo
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42, stratify=y_train_full
)

# =========================
# 3. Normalización (Crucial: Scaler solo aprende de Train)
# =========================
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test) # Aplicamos la misma escala al test

# =========================
# 4. Tensores
# =========================
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# =========================
# 5. DataLoaders
# =========================
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

# =========================
# 6. Modelo MLP (Arquitectura exacta del Paper)
# =========================
class MLP_Paper(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 128),
            nn.PReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.PReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.4),

            nn.Linear(64, 32),
            nn.PReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),

            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# =========================
# 7. Inicialización
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP_Paper(input_dim=X_train.shape[1], output_dim=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# =========================
# 8 y 9. Entrenamiento
# =========================
best_val_acc = 0
patience = 10
counter = 0
best_model = None

for epoch in range(300):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            _, preds = torch.max(outputs, 1)
            total += yb.size(0)
            correct += (preds == yb).sum().item()

    val_acc = correct / total
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model = model.state_dict().copy()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping en epoch {epoch+1}")
        break

# =========================
# 10. Evaluación Final (El test real)
# =========================
if best_model:
    model.load_state_dict(best_model)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)
        total += yb.size(0)
        correct += (preds == yb).sum().item()

print(f"\n--- Resultado Final (Opción 2: Sin Mezcla) ---")
print(f"Test Accuracy: {(correct / total):.4f}")

Modo Riguroso: El set de Test es totalmente independiente.
Epoch 10, Val Acc: 0.9866
Epoch 20, Val Acc: 0.9860
Epoch 30, Val Acc: 0.9193
Epoch 40, Val Acc: 0.9812
Early stopping en epoch 47

--- Resultado Final (Opción 2: Sin Mezcla) ---
Test Accuracy: 0.9288
